# Vietnamese Labor Law Retrieval API

Runs the verified 768-dimensional index with `truro7/vn-law-embedding` and `BAAI/bge-reranker-v2-m3` on a Kaggle GPU. The notebook starts a protected FastAPI service, exposes a temporary Cloudflare URL, runs the complete benchmark suite, and prints failed cases.

In [ ]:
# 1. Install runtime dependencies without replacing Kaggle's CUDA-enabled PyTorch.
!pip install -q chromadb==1.5.9 rank-bm25==0.2.2 sentence-transformers==6.1.0 requests==2.32.4 fastapi uvicorn

import torch
assert torch.cuda.is_available(), "Enable a GPU accelerator in Kaggle settings."
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Extract the private runtime bundle and create a Linux index pointer.
import json
import os
import shutil
import secrets
from pathlib import Path
from kaggle_secrets import UserSecretsClient

input_root = Path("/kaggle/input")
archives = list(input_root.rglob("labor-rag-runtime.zip"))
runtime_root = Path("/kaggle/working/labor-rag-runtime")
if runtime_root.exists():
    shutil.rmtree(runtime_root)
runtime_root.mkdir(parents=True)
if archives:
    if len(archives) != 1:
        raise RuntimeError(f"Expected one runtime archive, found {archives}")
    shutil.unpack_archive(str(archives[0]), str(runtime_root))
else:
    pointers = list(input_root.rglob("vectorstore_versions/current.json"))
    if len(pointers) != 1:
        raise RuntimeError(f"Expected one extracted runtime input, found {pointers}")
    extracted_root = pointers[0].parents[2]
    shutil.copytree(extracted_root, runtime_root, dirs_exist_ok=True)

legal_data = runtime_root / "legal_data"
pointer_source = json.loads(
    (legal_data / "vectorstore_versions/current.json").read_text(encoding="utf-8")
)
source_index = legal_data / "vectorstore_versions" / pointer_source["version"]
working_index = Path("/kaggle/working/labor-rag-index")
if working_index.exists():
    shutil.rmtree(working_index)
shutil.copytree(source_index, working_index)

pointer_source["index_dir"] = str(working_index)
working_pointer = Path("/kaggle/working/current.json")
working_pointer.write_text(
    json.dumps(pointer_source, ensure_ascii=False, indent=2), encoding="utf-8"
)

try:
    api_key = UserSecretsClient().get_secret("LEGAL_RAG_API_KEY")
except Exception:
    api_key = None
if not api_key:
    api_key = secrets.token_urlsafe(32)
    print("Generated ephemeral LEGAL_RAG_API_KEY for this private session:", api_key)

runtime_env = os.environ.copy()
runtime_env["LEGAL_RAG_CURRENT_POINTER"] = str(working_pointer)
runtime_env["LEGAL_RAG_API_KEY"] = api_key
runtime_env["LEGAL_RAG_REQUIRE_CUDA"] = "1"
print("Runtime root:", runtime_root)
print("Index:", working_index)
print("Index size MB:", round(sum(p.stat().st_size for p in working_index.rglob("*") if p.is_file()) / 2**20, 2))

In [ ]:
# 3. Start the GPU service once, then wait for the real health check.
import subprocess
import sys
import time
import requests

scripts = legal_data / "scripts"
service_log_path = Path("/kaggle/working/retrieval-service.log")
service_log = service_log_path.open("w", encoding="utf-8")
service = subprocess.Popen(
    [sys.executable, str(scripts / "kaggle_retrieval_api.py"), "--port", "9999"],
    cwd=str(scripts),
    env=runtime_env,
    stdout=service_log,
    stderr=subprocess.STDOUT,
)

health_url = "http://127.0.0.1:9999/health"
health_payload = None
for _ in range(120):
    if service.poll() is not None:
        service_log.flush()
        raise RuntimeError(service_log_path.read_text(encoding="utf-8")[-8000:])
    try:
        response = requests.get(health_url, timeout=5)
        if response.ok:
            health_payload = response.json()
            break
    except requests.RequestException:
        pass
    time.sleep(5)

if not health_payload:
    raise RuntimeError("Service did not become healthy within 10 minutes.")
print(json.dumps(health_payload, ensure_ascii=False, indent=2))

In [ ]:
# 4. Open a temporary Cloudflare tunnel for backend development.
import re

cloudflared = Path("/kaggle/working/cloudflared")
if not cloudflared.exists():
    url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    with requests.get(url, stream=True, timeout=120) as download:
        download.raise_for_status()
        with cloudflared.open("wb") as handle:
            for block in download.iter_content(1024 * 1024):
                handle.write(block)
    cloudflared.chmod(0o755)

tunnel_log_path = Path("/kaggle/working/cloudflared.log")
tunnel_log = tunnel_log_path.open("w", encoding="utf-8")
tunnel = subprocess.Popen(
    [str(cloudflared), "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:9999"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(60):
    time.sleep(1)
    tunnel_log.flush()
    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        tunnel_log_path.read_text(encoding="utf-8"),
    )
    if match:
        public_url = match.group(0)
        break

if not public_url:
    raise RuntimeError(tunnel_log_path.read_text(encoding="utf-8")[-8000:])
print("Public health:", f"{public_url}/health")
print("Public retrieve:", f"{public_url}/retrieve")
print("Keep LEGAL_RAG_API_KEY secret; send it in the X-API-Key header.")

In [ ]:
# 5. Run the complete evaluation suite through the HTTP service on GPU. Exit code 1 means thresholds failed, not a crashed run.
report_path = Path("/kaggle/working/retrieval_report.gpu.json")
benchmark = subprocess.run(
    [
        sys.executable,
        "-X",
        "utf8",
        str(scripts / "evaluate_retrieval_v2.py"),
        "--api-url",
        "http://127.0.0.1:9999",
        "--cases",
        str(legal_data / "eval/retrieval_cases.jsonl"),
        "--report",
        str(report_path),
    ],
    cwd=str(scripts),
    env=runtime_env,
    text=True,
)
if not report_path.exists():
    raise RuntimeError(f"Benchmark crashed with exit code {benchmark.returncode}")
report = json.loads(report_path.read_text(encoding="utf-8"))
print(json.dumps({
    "passed": report["passed"],
    "metrics": report["metrics"],
    "checks": report["checks"],
}, ensure_ascii=False, indent=2))

In [ ]:
# 6. Analyze failures and download the complete report.
failed = [row for row in report["cases"] if not row["passed"]]
print(f"Failed cases: {len(failed)}/{report['case_count']}")
for row in failed:
    print("-" * 80)
    print(row["id"], "|", row.get("category"), "| accepted:", row["actual_accepted"])
    print("query:", row["query"])
    print("expected docs:", row["expected_doc_numbers"])
    print("returned docs:", row["returned_doc_numbers"])
    print("expected articles:", row["expected_article_numbers"])
    print("returned articles:", row["returned_article_numbers"])
    print("doc recall:", row["doc_recall_at_k"], "article recall:", row["article_recall_at_k"])

from IPython.display import FileLink
display(FileLink(str(report_path)))
display(FileLink(str(service_log_path)))

# Stop ephemeral processes and remove copied runtime/index files so Kaggle
# exports only the small benchmark artifacts. The private Dataset is unchanged.
for process in (tunnel, service):
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
service_log.close()
tunnel_log.close()
for copied_dir in (working_index, runtime_root):
    if copied_dir.exists():
        shutil.rmtree(copied_dir)
for copied_file in (cloudflared, working_pointer):
    if copied_file.exists():
        copied_file.unlink()